# Text-to-SQL — Data Prep

Loads [`b-mc2/sql-create-context`](https://huggingface.co/datasets/b-mc2/sql-create-context) (78k schema + question + SQL examples), formats it into chat-style messages, and splits into train/val/test. Run in Colab, CPU only.

In [ ]:
!pip install -q -U datasets huggingface_hub

In [ ]:
from datasets import load_dataset
import os, shutil

PROJECT_DIR = '/content/text-to-sql-llm'
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)

## Load the raw dataset

In [ ]:
raw = load_dataset("b-mc2/sql-create-context")
print(raw)
print(raw["train"][0])

## Format into chat messages

Matches the Qwen2.5-Instruct chat template used directly by `SFTTrainer` later.

In [ ]:
SYSTEM_PROMPT = (
    "You are a precise text-to-SQL assistant. Given a database schema and a "
    "natural language question, output ONLY the SQL query that answers it. "
    "No explanation, no markdown, just the query."
)

def format_example(example):
    user_msg = f"Schema:\n{example['context']}\n\nQuestion: {example['question']}"
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": example["answer"]},
        ]
    }

formatted = raw["train"].map(format_example, remove_columns=raw["train"].column_names)
print(formatted[0]["messages"])

## Shuffle, subset, and split

90/5/5 train/val/test split on a 10k subset — fast to fine-tune on a free T4.

In [ ]:
formatted = formatted.shuffle(seed=42)

SUBSET_SIZE = 10000
subset = formatted.select(range(min(SUBSET_SIZE, len(formatted))))

split_a = subset.train_test_split(test_size=0.1, seed=42)
split_b = split_a["test"].train_test_split(test_size=0.5, seed=42)

train_ds, val_ds, test_ds = split_a["train"], split_b["train"], split_b["test"]
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

## Save and download

In [ ]:
train_ds.save_to_disk(f"{PROJECT_DIR}/data/train")
val_ds.save_to_disk(f"{PROJECT_DIR}/data/val")
test_ds.save_to_disk(f"{PROJECT_DIR}/data/test")

shutil.make_archive("/content/text-to-sql-data", "zip", f"{PROJECT_DIR}/data")

from google.colab import files
files.download("/content/text-to-sql-data.zip")